# HW#2 — การวิเคราะห์ชุดข้อมูล Titanic (Data Quality Assessment & EDA)

**รหัสนักศึกษา:** 6710503739  
**ไฟล์ส่ง:** `6710503739_hw_2_1.ipynb`

## สถานการณ์ทางธุรกิจ (Business Scenario)
ปี ค.ศ. 1912 หลังเหตุการณ์เรือ RMS Titanic อับปาง บริษัท **White Star Line** มอบหมายให้ทีมวิเคราะห์ข้อมูลตรวจสอบบันทึกข้อมูลผู้โดยสาร เพื่อตอบคำถามว่า
1. คุณลักษณะใดบ้างที่สัมพันธ์กับการรอดชีวิตของผู้โดยสาร
2. มีปัญหาด้านคุณภาพข้อมูล (data quality) ใดที่ต้องแก้ไขก่อนสร้างโมเดลทำนายหรือไม่

สมุดบันทึกนี้ตอบคำถาม **Question 2–6** ตามลำดับ พร้อมกราฟและคำอธิบายประกอบ

## เตรียมไลบรารีและโหลดข้อมูล

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy.stats import chi2_contingency, pearsonr

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

df = pd.read_csv("titanic_data.csv")
df.columns = df.columns.str.strip()  # ตัดช่องว่างหัว-ท้ายชื่อคอลัมน์กันพลาด
print("ขนาดข้อมูล (แถว, คอลัมน์):", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# สถิติเชิงพรรณนาของทุกคอลัมน์ (ทั้งตัวเลขและหมวดหมู่)
df.describe(include='all')

---
## Question 2: Data Quality Assessment
### Q2.1 การวิเคราะห์ข้อมูลสูญหาย (Missing Data Analysis)

In [ ]:
# จำนวนและเปอร์เซ็นต์ค่าที่สูญหายในแต่ละคอลัมน์
missing = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2)
})
missing = missing[missing.missing_count > 0].sort_values('missing_pct', ascending=False)
missing

In [ ]:
# Heatmap แสดงตำแหน่งค่าที่สูญหาย (แถบสว่าง = ค่าหาย)
plt.figure(figsize=(10, 6))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Heatmap')
plt.xlabel('Columns')
plt.show()

In [ ]:
# missingno: matrix แสดงรูปแบบการหายของข้อมูล
msno.matrix(df)
plt.show()

In [ ]:
# missingno: heatmap ความสัมพันธ์ (nullity correlation) ระหว่างคอลัมน์ที่มีค่าหาย
msno.heatmap(df)
plt.show()

#### สรุปผล Q2.1
**คอลัมน์ที่มีค่าสูญหาย และเปอร์เซ็นต์:**
- `Cabin` — สูญหาย **77.10%** (มากที่สุด)
- `Age` — สูญหาย **19.87%**
- `Embarked` — สูญหาย **0.22%** (เพียง 2 แถว)

คอลัมน์อื่น ๆ ไม่มีค่าสูญหาย

**เหมาะสมหรือไม่ที่จะลบคอลัมน์ `Cabin` ทิ้งทั้งหมด?**  
การลบทิ้ง *มีเหตุผลรองรับ* เพราะข้อมูลหายถึง 77% การเติมค่า (imputation) จึงเสี่ยงสร้างข้อมูลปลอมจำนวนมาก แต่ **ไม่ควรลบทิ้งทันทีโดยไม่พิจารณา** เพราะตัวอักษรตัวแรกของ Cabin (เช่น C, E, B) บอก *ชั้นดาดฟ้า (deck)* ซึ่งสัมพันธ์กับตำแหน่งบนเรือและโอกาสรอดชีวิต ทางเลือกที่ดีกว่าคือ **สกัดฟีเจอร์ใหม่** เช่น `HasCabin` (มี/ไม่มีข้อมูลห้อง) หรือ `Deck` (ตัวอักษรแรก) แล้วค่อยลบคอลัมน์ดิบทิ้ง — เก็บสัญญาณที่เป็นประโยชน์ไว้

**รูปแบบที่สังเกตได้จาก heatmap:**
- ค่าที่หายของ `Cabin` กระจายเกือบทั้งคอลัมน์ ส่วนใหญ่เป็นผู้โดยสารชั้น 2–3
- `Age` หายแบบกระจัดกระจาย (ไม่เป็นบล็อก) ลักษณะใกล้เคียง **MAR/MCAR** ยังพอเติมค่าได้
- จาก nullity heatmap ค่าสหสัมพันธ์การหายระหว่างคอลัมน์อยู่ในระดับต่ำ กล่าวคือการที่ `Age` หาย ไม่ได้บ่งชี้ชัดว่า `Cabin` จะหายด้วย — การหายของแต่ละคอลัมน์ค่อนข้างเป็นอิสระต่อกัน

### Q2.2 การตรวจจับค่าผิดปกติ (Outlier Detection)
ใช้วิธี **IQR (Interquartile Range)** กับตัวแปรเชิงตัวเลข `Age`, `Fare`, `SibSp`  
ขอบเขต: ค่าที่ต่ำกว่า `Q1 - 1.5*IQR` หรือสูงกว่า `Q3 + 1.5*IQR` ถือเป็น outlier

In [ ]:
def iqr_outliers(series):
    s = series.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < low) | (s > high)
    return {
        'Q1': round(q1, 2), 'Q3': round(q3, 2), 'IQR': round(iqr, 2),
        'lower_bound': round(low, 2), 'upper_bound': round(high, 2),
        'n_outliers': int(mask.sum()),
        'pct_outliers': round(mask.mean() * 100, 2),
    }

summary = pd.DataFrame({col: iqr_outliers(df[col]) for col in ['Age', 'Fare', 'SibSp']}).T
summary

In [ ]:
# Box plot ของทั้ง 3 ตัวแปร (จุดเลยหนวด = outlier ตามนิยาม IQR)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['Age', 'Fare', 'SibSp']):
    sns.boxplot(y=df[col], ax=ax, color='skyblue')
    ax.set_title(f'Box plot: {col}')
plt.tight_layout()
plt.show()

#### สรุปผล Q2.2
**จำนวน outlier ที่ตรวจพบด้วย IQR:** `Age` ≈ 11 ค่า (สูงเกิน 64.8 ปี), `Fare` ≈ 116 ค่า (สูงเกิน 65.6 สูงสุด 512.3), `SibSp` ≈ 46 ค่า (มากกว่า 2.5 คน)

**1) ทำไมเลือก IQR แทน Z-score?**  
เพราะ `Fare` และ `SibSp` **เบ้ขวาอย่างรุนแรง** (Fare skew ≈ 4.79) ไม่ได้กระจายแบบปกติ Z-score ตั้งอยู่บนสมมติฐานการแจกแจงปกติและใช้ค่าเฉลี่ย/ส่วนเบี่ยงเบนมาตรฐานซึ่ง *ถูกดึงโดยค่าสุดโต่งเอง* ทำให้ตรวจจับได้ไม่แม่น ส่วน IQR อาศัยควอไทล์ (robust ต่อค่าสุดโต่ง) จึงเหมาะกับข้อมูลเบ้มากกว่า

**2) outlier ทั้งหมดเป็น "ข้อมูลเสีย" หรือไม่?**  
ไม่ — ส่วนใหญ่เป็น **ค่าจริงที่ถูกต้อง** เช่น ผู้โดยสารอายุ 80 ปี, ค่าโดยสาร 512.33 ปอนด์ของห้องสวีทชั้นหนึ่ง, หรือครอบครัวใหญ่ที่มีพี่น้อง/คู่สมรส 8 คน ล้วนเป็นค่าที่เกิดขึ้นได้จริง ไม่ใช่ความผิดพลาดในการบันทึก จึงควรเรียกว่า *extreme values* มากกว่า *bad data*

**3) ควรลบ outlier ของ `Fare` ก่อนสร้างโมเดลทำนายหรือไม่?**  
**ไม่ควรลบทิ้ง** เพราะค่าโดยสารสูงสัมพันธ์โดยตรงกับชั้นโดยสารและฐานะ ซึ่งเป็น *ตัวทำนายการรอดชีวิตที่สำคัญ* การลบทิ้งจะทำให้เสียข้อมูลผู้โดยสารชั้นหนึ่งที่มีโอกาสรอดสูง ทางเลือกที่ดีกว่าคือ **แปลงค่า** เช่น log-transform หรือ winsorize/capping เพื่อลดอิทธิพลของค่าสุดโต่งโดยไม่ทิ้งข้อมูล

**4) เทคนิคตรวจจับ outlier เพิ่มเติม (นอกจาก IQR):**  
เช่น **Isolation Forest** (โมเดล ensemble) — ข้อดีคือทำงานกับ *หลายตัวแปรพร้อมกัน* จับ outlier ที่เห็นได้เฉพาะ เมื่อพิจารณาหลายมิติ (multivariate) ไม่ต้องสมมติการแจกแจง และปรับขนาดกับข้อมูลใหญ่ได้ดี

**5) ผลกระทบของ outlier ต่อการวิเคราะห์:**  
ดึงค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐานให้บิดเบือน (mean ของ Fare ≈ 32 แต่ median เพียง 14.5) ทำให้ค่าสหสัมพันธ์และโมเดลเชิงเส้นไวต่อค่าเหล่านี้ จึงควรใช้สถิติที่ทนทาน (median, IQR) และพิจารณาการแปลงค่าเสมอ

---
## Question 3: Univariate Analysis
### Q3.1 ตัวแปรเชิงตัวเลข (Numerical Variables)

In [ ]:
# Histogram ของ Age และ Fare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Age'].dropna(), bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title(f"Age (skewness = {df['Age'].skew():.2f})")
sns.histplot(df['Fare'].dropna(), bins=40, kde=True, ax=axes[1], color='indianred')
axes[1].set_title(f"Fare (skewness = {df['Fare'].skew():.2f})")
plt.tight_layout()
plt.show()

In [ ]:
# Box plot เปรียบเทียบ Age, Fare, SibSp
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['Age', 'Fare', 'SibSp']):
    sns.boxplot(y=df[col], ax=ax, color='lightgreen')
    ax.set_title(f'{col}')
plt.tight_layout()
plt.show()

#### สรุปผล Q3.1
**Histogram:**
- `Age` มีค่า skewness ≈ 0.39 — **เบ้ขวาเล็กน้อย** รูปทรงใกล้ระฆังคว่ำ กระจุกช่วง 20–40 ปี (mean 29.7 / median 28.0 ใกล้กัน) ถือว่า *ค่อนข้างสมมาตร*
- `Fare` มีค่า skewness ≈ 4.79 — **เบ้ขวาอย่างรุนแรง** ผู้โดยสารส่วนใหญ่จ่ายค่าโดยสารต่ำ มีหางยาวไปทางค่าสูง (mean 32.2 >> median 14.5) ไม่ใช่การแจกแจงปกติ

**Box plot:**
- `Age` การกระจายสมมาตรที่สุด มี outlier ด้านสูงเพียงไม่กี่จุด
- `Fare` กล่องถูกอัดที่ค่าต่ำ มี outlier ด้านสูงจำนวนมาก (หางขวายาว) — ยืนยันการเบ้ขวา
- `SibSp` ค่ามัธยฐาน = 0 ผู้โดยสารส่วนใหญ่เดินทางคนเดียว/ไม่มีพี่น้อง ค่าที่ ≥ 3 กลายเป็น outlier
- แนวโน้มสู่ศูนย์กลาง: median ของ Age สูงสุด (~28), Fare (~14.5), SibSp (0) ตามลำดับ

### Q3.2 ตัวแปรเชิงหมวดหมู่ (Categorical Variables)

In [ ]:
# Bar chart ของตัวแปรหมวดหมู่หลัก
cat_cols = ['Survived', 'Sex', 'Pclass', 'Embarked']
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(x=col, data=df, ax=ax, palette='Set2', hue=col, legend=False)
    ax.set_title(f'Distribution of {col}')
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width()/2, p.get_height()),
                    ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
# Pie chart: สัดส่วนการรอดชีวิต และสัดส่วนเพศ
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['Survived'].value_counts().plot.pie(
    autopct='%1.1f%%', labels=['Died (0)', 'Survived (1)'],
    colors=['#d95f5f', '#5f9ed9'], ax=axes[0])
axes[0].set_title('Survived'); axes[0].set_ylabel('')
df['Sex'].value_counts().plot.pie(
    autopct='%1.1f%%', colors=['#66c2a5', '#fc8d62'], ax=axes[1])
axes[1].set_title('Sex'); axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

#### สรุปผล Q3.2
- **Survived:** ผู้โดยสารรอดชีวิตเพียง **38.4%** (342 คน) ส่วนใหญ่เสียชีวิต — ข้อมูลไม่สมดุล (imbalanced)
- **Sex:** เพศชายมากกว่าเพศหญิงชัดเจน (~65% ชาย / ~35% หญิง)
- **Pclass:** ผู้โดยสารชั้น 3 มากที่สุด (491 คน) มากกว่าชั้น 1 และ 2 รวมกัน
- **Embarked:** ท่าเรือ Southampton (S) เป็นจุดขึ้นเรือหลัก รองลงมาคือ Cherbourg (C) และ Queenstown (Q)
- ข้อสังเกต: ความไม่สมดุลของ Survived สำคัญต่อการเลือกเมตริกประเมินโมเดลภายหลัง (accuracy อย่างเดียวอาจหลอกตา)

---
## Question 4: Bivariate Analysis – Numerical Variables
### Q4.1 การวิเคราะห์สหสัมพันธ์ (Correlation Analysis)

In [ ]:
# Pearson correlation ระหว่าง Age และ Fare (ตัดแถวที่ Age หายออก)
sub = df[['Age', 'Fare']].dropna()
r, p_val = pearsonr(sub['Age'], sub['Fare'])
print(f'Pearson r = {r:.4f}')
print(f'p-value  = {p_val:.4g}')

In [ ]:
# Scatter plot Age vs Fare (แยกสีตามการรอดชีวิต)
plt.figure(figsize=(9, 6))
sns.scatterplot(x='Age', y='Fare', hue='Survived', data=df, alpha=0.6, palette='coolwarm')
plt.title('Age vs Fare')
plt.show()

#### สรุปผล Q4.1
- **ค่า Pearson r ≈ 0.096** — เป็นความสัมพันธ์เชิงบวกที่ **อ่อนมาก (very weak)** แปลว่าอายุกับค่าโดยสารแทบไม่มีความสัมพันธ์เชิงเส้นต่อกัน (แม้ p-value จะมีนัยสำคัญ แต่ขนาดความสัมพันธ์เล็กมาก)
- **Scatter plot:** จุดกระจายกระจุกที่ค่าโดยสารต่ำในทุกช่วงอายุ ไม่เห็นแนวโน้มเชิงเส้นชัดเจน
- สังเกตกลุ่มค่าโดยสารสูงผิดปกติ (Fare > 200) ซึ่งส่วนใหญ่เป็นจุดสีของผู้ **รอดชีวิต** — สะท้อนว่า *ค่าโดยสาร (ฐานะ/ชั้น)* สัมพันธ์กับการรอดชีวิตมากกว่าที่ *อายุ* สัมพันธ์กับค่าโดยสาร

---
## Question 5: Bivariate Analysis – Categorical Variables
### Q5.1 ตารางความสัมพันธ์ (Contingency Table): Survived × Pclass

In [ ]:
# ตารางไขว้จำนวนคน
ct = pd.crosstab(df['Pclass'], df['Survived'], margins=True)
ct.columns = ['Died(0)', 'Survived(1)', 'All']
print(ct)

# เปอร์เซ็นต์การรอดชีวิตในแต่ละชั้น
surv_pct = (df.groupby('Pclass')['Survived'].mean() * 100).round(2)
print('\nSurvival rate by Pclass (%):')
print(surv_pct)

In [ ]:
# กราฟแท่งเปอร์เซ็นต์การรอดชีวิตต่อชั้นโดยสาร
plt.figure(figsize=(7, 5))
ax = sns.barplot(x=surv_pct.index, y=surv_pct.values, palette='viridis', hue=surv_pct.index, legend=False)
for i, v in enumerate(surv_pct.values):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center')
plt.title('Survival Rate by Passenger Class')
plt.ylabel('Survival Rate (%)'); plt.xlabel('Pclass')
plt.ylim(0, 75)
plt.show()

#### สรุปผล Q5.1
| ชั้นโดยสาร | อัตราการรอดชีวิต |
|:--:|:--:|
| ชั้น 1 | **62.96%** |
| ชั้น 2 | 47.28% |
| ชั้น 3 | **24.24%** |

อัตราการรอดชีวิตลดลงชัดเจนตามชั้นโดยสารที่ต่ำลง — ผู้โดยสารชั้น 1 มีโอกาสรอดสูงกว่าชั้น 3 ถึงราว 2.6 เท่า ชี้ว่า **ชั้นโดยสาร (สถานะทางเศรษฐกิจ) สัมพันธ์อย่างมากกับการรอดชีวิต** สอดคล้องกับการเข้าถึงเรือชูชีพ และตำแหน่งห้องพักบนเรือ

### Q5.2 การทดสอบทางสถิติ (Chi-Square Test of Independence)

In [ ]:
# ทดสอบไคสแควร์ความเป็นอิสระระหว่าง Survived และ Pclass
observed = pd.crosstab(df['Survived'], df['Pclass'])
chi2, p, dof, expected = chi2_contingency(observed)
print(f'Chi-square statistic = {chi2:.3f}')
print(f'p-value              = {p:.3e}')
print(f'degrees of freedom   = {dof}')
print('\nExpected frequencies:')
print(pd.DataFrame(expected, index=['Died', 'Survived'], columns=[1, 2, 3]).round(1))

#### สรุปผล Q5.2
**สมมติฐาน:**
- H₀ (null): การรอดชีวิต (`Survived`) และชั้นโดยสาร (`Pclass`) **เป็นอิสระต่อกัน** (ไม่มีความสัมพันธ์)
- H₁ (alternative): การรอดชีวิตและชั้นโดยสาร **มีความสัมพันธ์กัน**

**ผลการทดสอบ:** χ² ≈ **102.89**, p-value ≈ **4.55 × 10⁻²³**, degrees of freedom = **2**

**ข้อสรุป:** เนื่องจาก p-value น้อยกว่า 0.05 อย่างมาก จึง **ปฏิเสธ H₀** — สรุปได้ว่าการรอดชีวิตกับชั้นโดยสาร **มีความสัมพันธ์กันอย่างมีนัยสำคัญทางสถิติ** ยืนยันข้อสังเกตจาก Q5.1 ว่าชั้นโดยสารเป็นปัจจัยสำคัญต่อการรอดชีวิต

---
## Question 6: Multi-variate Analysis
### Q6.1 Pairplot: Age, Fare, SibSp

In [ ]:
# Pairplot แยกสีตามการรอดชีวิต เพื่อดูความสัมพันธ์หลายตัวแปรพร้อมกัน
pair_df = df[['Age', 'Fare', 'SibSp', 'Survived']].dropna()
sns.pairplot(pair_df, vars=['Age', 'Fare', 'SibSp'], hue='Survived',
             palette='coolwarm', diag_kind='kde', plot_kws={'alpha': 0.5})
plt.suptitle('Pairplot: Age, Fare, SibSp (by Survived)', y=1.02)
plt.show()

#### สรุปผล Q6.1
- **ความสัมพันธ์ระหว่างตัวแปร:** ทุกคู่ (Age–Fare, Age–SibSp, Fare–SibSp) แสดงความสัมพันธ์เชิงเส้นที่ *อ่อนมาก* ไม่มีคู่ใดสัมพันธ์กันชัดเจน (สอดคล้องกับ Q4)
- **การกระจายแนวทแยง (diagonal):** ยืนยันว่า `Fare` และ `SibSp` เบ้ขวารุนแรง ส่วน `Age` ค่อนข้างสมมาตร
- **สัญญาณการรอดชีวิต:** ในมิติ Fare ผู้ที่จ่ายค่าโดยสารสูง (จุดสีของผู้รอด) มีสัดส่วนรอดชีวิตมากกว่า — Fare ให้พลังแยกแยะการรอดชีวิตได้ดีกว่า Age หรือ SibSp
- **outlier ที่เห็นจาก pairplot:** จุด `Fare` > 500 (ห้องสวีทชั้นหนึ่ง), `Age` = 80 ปี, และ `SibSp` = 8 (ครอบครัวใหญ่) โดดออกจากกลุ่มหลักในทุกแกนที่เกี่ยวข้อง

---
### สรุปภาพรวม (Overall Conclusion)
1. **คุณภาพข้อมูล:** ต้องจัดการค่าสูญหายใน `Cabin` (77%), `Age` (20%), `Embarked` (2 แถว) และมี extreme values ใน `Fare`/`SibSp` ที่เป็นค่าจริง — ควรแปลงค่ามากกว่าลบทิ้ง
2. **ปัจจัยที่สัมพันธ์กับการรอดชีวิต:** ชั้นโดยสาร (Pclass) และค่าโดยสาร (Fare) ซึ่งสะท้อนฐานะ มีความสัมพันธ์กับการรอดชีวิตอย่างมีนัยสำคัญ (ยืนยันด้วย Chi-square) ขณะที่อายุมีบทบาทน้อยกว่า